# Model training: RF + XGBoost across all 6 datasets

One CONFIG dict per (dataset x model) run -- 12 runs total. Every run uses the **same search method (Optuna, TPE sampler)** and the **same fixed budget (N_TRIALS=50, CV=5)** by default (see `DEFAULT_N_TRIALS`/`DEFAULT_CV_FOLDS` and the `_suggest_rf_params`/`_suggest_xgb_params` search spaces in `src/training.py`). Only `dataset_name`, `domain`, and `model_type` change per iteration.

If a specific run's runtime genuinely isn't workable at the fixed budget, `get_default_search_budget(n_rows)` is available as an **opt-in** fallback (row-count tiers down to fewer trials/folds) -- but it is never applied automatically, so any use of it should be an explicit, visible entry in `EXTRA_CONFIG_OVERRIDES` below, not a silent default.

Requires: `pip install optuna imbalanced-learn xgboost skops` (skops and huggingface_hub are optional -- see notes inline).

In [1]:
import sys
sys.path.insert(0, "..")
from src.training import run_training, aggregate_model_results, get_default_search_budget


In [2]:
DATASETS = [
    # ("healthcare", "pima_diabetes"),
    # ("healthcare", "breast_cancer_wisconsin"),
    # ("healthcare", "heart_disease_uci"),
    ("finance", "loan_default"),
    ("finance", "financial_distress"),
    ("finance", "credit_card_fraud_2023"),
]
MODEL_TYPES = ["rf", "xgb"]

# Per-run exceptions to the shared protocol, if any are ever needed -- keep empty
# by default so every run genuinely uses the same search protocol.
#
# search_sample_size caps how many rows the CV *search* step trains on (a
# stratified subsample) purely for runtime -- it does NOT change the search
# space, CV folds, n_trials, or scoring metric, and the final refit in step 4
# always uses the FULL balanced training set regardless. Used here for the
# two largest datasets (loan_default: ~360k balanced rows, credit_card_fraud_2023:
# ~1.1M balanced rows), tuning on a stratified 50k-row sample instead of the full set.
#
# If that's still too slow, the next lever is the OPT-IN tiered budget, e.g.:
#   EXTRA_CONFIG_OVERRIDES[("finance", "loan_default", "rf")].update(
#       get_default_search_budget(200_000)  # -> {"n_trials": 20, "cv_folds": 3}
#   )
# Apply it explicitly per run, not automatically, and note it in the writeup.
EXTRA_CONFIG_OVERRIDES = {
    ("finance", "loan_default", "rf"): {"search_sample_size": 50_000},
    ("finance", "loan_default", "xgb"): {"search_sample_size": 50_000},
    ("finance", "credit_card_fraud_2023", "rf"): {"search_sample_size": 50_000},
    ("finance", "credit_card_fraud_2023", "xgb"): {"search_sample_size": 50_000},
}

# Set to True (and fill in hub_repo_id per run below) once you're ready to publish.
PUSH_TO_HUB = False
HUB_NAMESPACE = "yourname"   # e.g. "yourname/xai-pima-rf"


In [3]:
all_results = {}

for domain, dataset_name in DATASETS:
    for model_type in MODEL_TYPES:
        print(f"\n{'='*70}\nTRAINING: {dataset_name} x {model_type}\n{'='*70}")

        config = {
            "dataset_name": dataset_name,
            "domain": domain,
            "model_type": model_type,
            # everything else (cv_folds, n_iter, scoring, random_state, param_distributions)
            # falls back to the fixed defaults in src/training.py
            "datasets_root": "../Datasets",
            "models_root": "../Models",
        }
        config.update(EXTRA_CONFIG_OVERRIDES.get((domain, dataset_name, model_type), {}))

        if PUSH_TO_HUB:
            config["push_to_hub"] = True
            config["hub_repo_id"] = f"{HUB_NAMESPACE}/xai-{dataset_name.replace('_', '-')}-{model_type}"

        result = run_training(config)
        all_results[(dataset_name, model_type)] = result
        print(f"Test metrics: {result['test_metrics']}")



TRAINING: loan_default x rf

------------------------------------------------------------
1. CONFIGURATION
------------------------------------------------------------
{
  "dataset_name": "loan_default",
  "domain": "finance",
  "model_type": "rf",
  "datasets_root": "../Datasets",
  "models_root": "../Models",
  "search_sample_size": 50000
}
Search budget: n_trials=50, cv_folds=5 (search_method=optuna)
target not given in config -- read 'Default' from dataset_info.json

------------------------------------------------------------
2. LOAD ARTIFACTS
------------------------------------------------------------


[I 2026-07-07 05:39:38,602] A new study created in memory with name: no-name-cb51a141-3fc1-4e15-8837-cfbfbccb27be


Loaded pre-balance train: (204277, 31)
Loaded balanced train:    (361110, 31)
Loaded test:               (51070, 31)
Feature columns (31): ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', "Education_Bachelor's", 'Education_High School', "Education_Master's", 'Education_PhD', 'EmploymentType_Full-time', 'EmploymentType_Part-time', 'EmploymentType_Self-employed', 'EmploymentType_Unemployed', 'MaritalStatus_Divorced', 'MaritalStatus_Married', 'MaritalStatus_Single', 'HasMortgage_No', 'HasMortgage_Yes', 'HasDependents_No', 'HasDependents_Yes', 'LoanPurpose_Auto', 'LoanPurpose_Business', 'LoanPurpose_Education', 'LoanPurpose_Home', 'LoanPurpose_Other', 'HasCoSigner_No', 'HasCoSigner_Yes']
Fitted transformers bundle found: True

------------------------------------------------------------
3. HYPERPARAMETER SEARCH
------------------------------------------------------------
search_sample_size=50000: searching on a strat

[I 2026-07-07 05:39:55,227] Trial 0 finished with value: 0.7151326204809784 and parameters: {'n_estimators': 250, 'max_depth': 5, 'min_samples_split': 3, 'min_samples_leaf': 18, 'max_features': 'log2'}. Best is trial 0 with value: 0.7151326204809784.
[I 2026-07-07 05:40:26,312] Trial 1 finished with value: 0.714206440952854 and parameters: {'n_estimators': 488, 'max_depth': 5, 'min_samples_split': 11, 'min_samples_leaf': 9, 'max_features': 'log2'}. Best is trial 0 with value: 0.7151326204809784.
[I 2026-07-07 05:41:00,745] Trial 2 finished with value: 0.7294168784984312 and parameters: {'n_estimators': 217, 'max_depth': 15, 'min_samples_split': 13, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 2 with value: 0.7294168784984312.
[I 2026-07-07 05:44:33,253] Trial 3 finished with value: 0.6994250338516332 and parameters: {'n_estimators': 480, 'max_depth': 5, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': None}. Best is trial 2 with value: 0.7294168784984312


Search completed in 5636.3s
Best CV roc_auc: 0.7337
Best params (pipeline-prefixed): {'clf__n_estimators': 449, 'clf__max_depth': 30, 'clf__min_samples_split': 9, 'clf__min_samples_leaf': 18, 'clf__max_features': 'sqrt'}

------------------------------------------------------------
4. REFIT FINAL MODEL
------------------------------------------------------------
Refitting on full balanced training set with params: {'n_estimators': 449, 'max_depth': 30, 'min_samples_split': 9, 'min_samples_leaf': 18, 'max_features': 'sqrt'}
Final refit completed in 119.24s on 361110 rows

------------------------------------------------------------
5. HELD-OUT TEST EVALUATION
------------------------------------------------------------
Test metrics:
  accuracy: 0.8597
  precision: 0.3647
  recall: 0.2802
  f1: 0.3169
  roc_auc: 0.7485
  pr_auc: 0.3016
Confusion matrix:
[[42244  2895]
 [ 4269  1662]]

------------------------------------------------------------
6. FIGURES
-------------------------------

...


Search completed in 5823.6s
Best CV roc_auc: 0.9999
Best params (pipeline-prefixed): {'clf__n_estimators': 235, 'clf__max_depth': 30, 'clf__min_samples_split': 12, 'clf__min_samples_leaf': 1, 'clf__max_features': 'log2'}

------------------------------------------------------------
4. REFIT FINAL MODEL
------------------------------------------------------------
Refitting on full balanced training set with params: {'n_estimators': 235, 'max_depth': 30, 'min_samples_split': 12, 'min_samples_leaf': 1, 'max_features': 'log2'}
Final refit completed in 237.82s on 454903 rows

------------------------------------------------------------
5. HELD-OUT TEST EVALUATION
------------------------------------------------------------
Test metrics:
  accuracy: 0.9998
  precision: 0.9996
  recall: 1.0000
  f1: 0.9998
  roc_auc: 1.0000
  pr_auc: 1.0000
Confusion matrix:
[[56841    22]
 [    0 56863]]

------------------------------------------------------------
6. FIGURES
-------------------------------

[I 2026-07-07 09:30:14,381] A new study created in memory with name: no-name-b8e51937-da0c-4640-bf7a-e55a0e264262


search_sample_size=50000: searching on a stratified subsample (50000 of 454903 rows). Final refit in step 4 still uses the full balanced training set.
model_type: xgb
search_method: Optuna TPE, n_trials=50
cv: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring: roc_auc

About to run 50 trials x 5 folds = 250 total fits. Optuna prints each trial's result to stdout as it completes (not silent).


[I 2026-07-07 09:30:21,438] Trial 0 finished with value: 0.9999491920000001 and parameters: {'n_estimators': 250, 'max_depth': 10, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2}. Best is trial 0 with value: 0.9999491920000001.
[I 2026-07-07 09:30:26,032] Trial 1 finished with value: 0.999884164 and parameters: {'n_estimators': 123, 'max_depth': 9, 'learning_rate': 0.07725378389307355, 'subsample': 0.8832290311184181, 'colsample_bytree': 0.608233797718321, 'min_child_weight': 10}. Best is trial 0 with value: 0.9999491920000001.
[I 2026-07-07 09:30:35,270] Trial 2 finished with value: 0.999038788 and parameters: {'n_estimators': 433, 'max_depth': 4, 'learning_rate': 0.01855998084649059, 'subsample': 0.6733618039413735, 'colsample_bytree': 0.7216968971838151, 'min_child_weight': 6}. Best is trial 0 with value: 0.9999491920000001.
[I 2026-07-07 09:30:41,133] Trial 3 finished with value: 0.999912864 and pa


Search completed in 529.0s
Best CV roc_auc: 1.0000
Best params (pipeline-prefixed): {'clf__n_estimators': 478, 'clf__max_depth': 6, 'clf__learning_rate': 0.1675629037403375, 'clf__subsample': 0.8364541994575864, 'clf__colsample_bytree': 0.7451940727299186, 'clf__min_child_weight': 1}

------------------------------------------------------------
4. REFIT FINAL MODEL
------------------------------------------------------------
Refitting on full balanced training set with params: {'n_estimators': 478, 'max_depth': 6, 'learning_rate': 0.1675629037403375, 'subsample': 0.8364541994575864, 'colsample_bytree': 0.7451940727299186, 'min_child_weight': 1}
Final refit completed in 18.05s on 454903 rows

------------------------------------------------------------
5. HELD-OUT TEST EVALUATION
------------------------------------------------------------
Test metrics:
  accuracy: 0.9998
  precision: 0.9997
  recall: 1.0000
  f1: 0.9998
  roc_auc: 1.0000
  pr_auc: 1.0000
Confusion matrix:
[[56844    1

## Cross-run aggregation
Run once all 12 runs above have completed. Produces `Models/summary/model_performance_summary.csv` and a comparison figure -- worth checking here before moving on to explanations, since explaining a poorly-fit model tells you little about the explainer.

In [5]:
summary = aggregate_model_results(models_root="../Models")
summary["summary_df"]


,domain,dataset_name,model_name,accuracy,precision,recall,f1,roc_auc,pr_auc,best_hyperparameters
0,finance,credit_card_fraud_2023,rf,0.999807,0.999613,1.000000,0.999807,0.999985,0.999963,"{""n_estimators"": 235, ""max_depth"": 30, ""min_sa..."
1,finance,credit_card_fraud_2023,xgb,0.999833,0.999666,1.000000,0.999833,0.999987,0.999984,"{""n_estimators"": 478, ""max_depth"": 6, ""learnin..."
2,finance,financial_distress,rf,0.862398,0.201613,0.925926,0.331126,0.943109,0.388724,"{""n_estimators"": 431, ""max_depth"": 5, ""min_sam..."
3,finance,financial_distress,xgb,0.927793,0.259259,0.518519,0.345679,0.919587,0.300865,"{""n_estimators"": 123, ""max_depth"": 9, ""learnin..."
4,finance,loan_default,rf,0.859722,0.364714,0.280223,0.316934,0.748451,0.301619,"{""n_estimators"": 449, ""max_depth"": 30, ""min_sa..."
5,finance,loan_default,xgb,0.885686,0.545633,0.093745,0.160000,0.751882,0.318648,"{""n_estimators"": 162, ""max_depth"": 3, ""learnin..."
6,healthcare,breast_cancer_wisconsin,rf,0.973684,1.000000,0.928571,0.962963,0.996528,0.994152,"{""max_depth"": 20, ""max_features"": ""log2"", ""min..."
7,healthcare,breast_cancer_wisconsin,xgb,0.982456,1.000000,0.952381,0.975610,0.993056,0.992063,"{""colsample_bytree"": 0.7077649335194086, ""lear..."
8,healthcare,heart_disease_uci,rf,0.858696,0.845455,0.911765,0.877358,0.923003,0.934763,"{""max_depth"": 10, ""max_features"": ""log2"", ""min..."
9,healthcare,heart_disease_uci,xgb,0.858696,0.845455,0.911765,0.877358,0.912482,0.916071,"{""colsample_bytree"": 0.6036788206466518, ""lear..."
